# Fase 1 · Sample: muestreo y selección de datos

El objetivo de esta fase es disponer de un conjunto de fragmentos de código y archivos de configuración, etiquetados como vulnerables o no vulnerables y asociados a una categoría de OWASP Top 10:2025 y, cuando corresponde, de OWASP API Security Top 10:2023.

El dataset se construye con dos scripts que pueden ejecutarse de nuevo en cualquier momento:

```
python scripts/descargar_fuentes.py     # clona OWASP Benchmark (Java y Python) en data/raw
python scripts/construir_dataset.py     # muestrea, etiqueta, deduplica y particiona
```

Este notebook no repite esa lógica; carga el resultado y verifica que cumple las condiciones pedidas. La descripción detallada de las fuentes, el diccionario de datos y la justificación del muestreo están en `docs/fase1_sample.md`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ))
pd.set_option("display.max_colwidth", 60)

df = pd.read_csv(RAIZ / "data" / "dataset_vulnerabilidades.csv", keep_default_na=False)
mapeo = pd.read_csv(RAIZ / "data" / "mapeo_cwe_owasp.csv", keep_default_na=False)
print(f"{len(df)} registros, {df.shape[1]} columnas")
df.drop(columns=["codigo", "sha256"]).head()

1541 registros, 17 columnas


,id,fuente,proyecto_origen,archivo_origen,lenguaje,tipo_archivo,framework,categoria_vuln,cwe,vulnerable,owasp2025,api2023,grupo,particion,n_lineas
0,VM00001,OWASP BenchmarkJava,BenchmarkJava,BenchmarkJava/src/main/java/org/owasp/benchmark/testcode...,java,java,servlet,inyeccion_ldap,CWE-90,1,A05:2025,,BenchmarkJava:BenchmarkTest00012,val,113
1,VM00002,OWASP BenchmarkJava,BenchmarkJava,BenchmarkJava/src/main/java/org/owasp/benchmark/testcode...,java,java,servlet,cookie_insegura,CWE-614,0,,,BenchmarkJava:BenchmarkTest00016,train,84
2,VM00003,OWASP BenchmarkJava,BenchmarkJava,BenchmarkJava/src/main/java/org/owasp/benchmark/testcode...,java,java,servlet,inyeccion_ldap,CWE-90,1,A05:2025,,BenchmarkJava:BenchmarkTest00021,train,103
3,VM00004,OWASP BenchmarkJava,BenchmarkJava,BenchmarkJava/src/main/java/org/owasp/benchmark/testcode...,java,java,servlet,limite_confianza,CWE-501,1,A06:2025,,BenchmarkJava:BenchmarkTest00031,train,61
4,VM00005,OWASP BenchmarkJava,BenchmarkJava,BenchmarkJava/src/main/java/org/owasp/benchmark/testcode...,java,java,servlet,inyeccion_sql,CWE-89,1,A05:2025,,BenchmarkJava:BenchmarkTest00033,train,79


## 1. Fuentes utilizadas

Se combinaron tres fuentes. Las dos primeras son los proyectos OWASP Benchmark para Java (v1.2) y para Python (v0.1). Cada uno contiene miles de casos de prueba pequeños, escritos expresamente para medir herramientas de análisis, y un archivo `expectedresults` que indica si cada caso es realmente vulnerable y qué CWE representa. Eso nos da etiquetas confiables tanto para ejemplos vulnerables como para falsos positivos intencionales (código que parece vulnerable pero no lo es).

Benchmark, sin embargo, solo cubre inyecciones, criptografía, cookies y algunos riesgos de diseño en aplicaciones web clásicas. No contiene autorización a nivel de objeto, SSRF, consumo de recursos, ni archivos `application.yml`, `Dockerfile` o `docker-compose.yml`. Para cubrir esas categorías, que son justamente las de una arquitectura de microservicios, el equipo escribió casos controlados en Spring Boot y FastAPI (tercera fuente). Cada patrón tiene una versión vulnerable y una versión corregida, y se instancia varias veces cambiando entidades, rutas y nombres de servicio.

In [2]:
resumen = (df.groupby(["fuente", "lenguaje"])["vulnerable"]
           .agg(registros="size", vulnerables="sum"))
resumen["no_vulnerables"] = resumen["registros"] - resumen["vulnerables"]
resumen

registros  vulnerables  no_vulnerables
fuente                       lenguaje                                        
Casos controlados del equipo config          108           54              54
                             java            195          105              90
                             python          162           88              74
OWASP BenchmarkJava          java            535          265             270
OWASP BenchmarkPython        python          541          260             281

## 2. Procedimiento de muestreo

En Benchmark las categorías están muy desbalanceadas: en Java hay 504 casos de inyección SQL y solo 35 de XPath. Si se tomaran todos, el modelo aprendería sobre todo a reconocer SQL y `weakrand`. Por eso se hizo un muestreo estratificado por la pareja (categoría, etiqueta) con un tope de 25 casos por estrato. Los estratos pequeños se toman completos y los grandes se reducen, con semilla fija (42) para que el resultado sea reproducible.

En los casos controlados cada plantilla se instancia cinco veces. Los valores que se varían (entidad, ruta, identificador, servicio y puerto) no se eligen al azar, sino recorriendo cada lista en ciclo con un contador separado para las plantillas vulnerables y otro para las seguras. Así ambas clases reciben la misma distribución de nombres. En una primera versión se eligieron al azar y la auditoría de sesgo (fase Modify) mostró que algunas entidades quedaban asociadas a una clase; por ejemplo, `Booking` aparecía el doble de veces en código vulnerable. Los archivos de dependencias (`pom.xml`, `requirements.txt`) se generan combinando al azar librerías con versiones afectadas por CVE conocidos o con versiones vigentes.

Después se eliminaron los duplicados exactos (mismo hash SHA-256 del código). El script informó 28, todos provenientes de plantillas controladas cuyo texto no depende de los marcadores que se varían.

In [3]:
print("Total:", len(df))
print("No vulnerables:", (df.vulnerable == 0).sum(), "(mínimo exigido: 100)")
print("Duplicados exactos restantes:", df.sha256.duplicated().sum())
pd.crosstab(df.categoria_vuln, df.vulnerable, margins=True, margins_name="total")

Total: 1541
No vulnerables: 769 (mínimo exigido: 100)
Duplicados exactos restantes: 0


vulnerable,0,1,total
categoria_vuln,,,
aleatorio_debil,50,50,100
asignacion_masiva,19,19,38
autenticacion,21,25,46
bfla,19,19,38
bola,20,25,45
cifrado_debil,25,25,50
configuracion,35,35,70
consumo_recursos,20,20,40
cookie_insegura,40,49,89


## 3. Correspondencia CWE–OWASP

La etiqueta OWASP de cada registro no se asignó a mano, sino a partir del CWE usando la tabla `data/mapeo_cwe_owasp.csv`. Así cada etiqueta es trazable hasta su origen.

Para OWASP Top 10:2025 se usó la lista oficial de CWE asociados a cada categoría publicada en top10.owasp.org/2025. Dos CWE que usamos no aparecen en esa lista: CWE-770 (asignación de recursos sin límites), que se ubicó en A06 Diseño inseguro porque la ausencia de límites es una decisión de diseño, y CWE-250 (ejecución con privilegios innecesarios, por ejemplo un contenedor que corre como root), que se ubicó en A02 Configuración incorrecta. OWASP API Security Top 10:2023 no publica una tabla CWE completa, así que se tomaron los CWE citados en las referencias de cada riesgo y, para el resto, se aplicó el criterio del equipo. La columna `fuente_*` de la tabla distingue ambos casos.

Un ejemplo no vulnerable no lleva etiqueta OWASP: su CWE indica contra qué debilidad está protegido el fragmento, no una debilidad presente.

In [4]:
mapeo

,cwe,nombre_cwe,owasp2025,api2023,fuente_owasp2025,fuente_api2023
0,CWE-22,Path Traversal,A01:2025,,oficial,
1,CWE-601,Open Redirect,A01:2025,,oficial,
2,CWE-639,Authorization Bypass Through User-Controlled Key,A01:2025,API1:2023,oficial,referencia oficial API1
3,CWE-285,Improper Authorization,A01:2025,API5:2023,oficial,referencia oficial API5
4,CWE-918,Server-Side Request Forgery,A01:2025,API7:2023,oficial,referencia oficial API7
5,CWE-16,Configuration,A02:2025,API8:2023,oficial,referencia oficial API8
6,CWE-942,Permissive Cross-domain Policy,A02:2025,API8:2023,oficial,referencia oficial API8
7,CWE-489,Active Debug Code,A02:2025,API8:2023,oficial,criterio del equipo
8,CWE-614,Sensitive Cookie Without Secure Attribute,A02:2025,,oficial,
9,CWE-611,XML External Entity (XXE),A02:2025,,oficial,


In [5]:
print("CWE del dataset sin mapeo:", set(df.cwe) - set(mapeo.cwe) or "ninguno")
vuln = df[df.vulnerable == 1]
print("Vulnerables sin categoría OWASP 2025:", (vuln.owasp2025 == "").sum())
print("No vulnerables con alguna etiqueta:", ((df.vulnerable == 0) & ((df.owasp2025 != "") | (df.api2023 != ""))).sum())

CWE del dataset sin mapeo: ninguno
Vulnerables sin categoría OWASP 2025: 0
No vulnerables con alguna etiqueta: 0


## 4. Partición en entrenamiento, validación y prueba

El enunciado recomienda 70/15/15 separando por proyecto de origen. Separar literalmente por proyecto no tiene sentido aquí, porque Benchmark Java es un único proyecto y quedaría entero en una sola partición. Lo que sí importa es la idea de fondo: que en prueba no haya variantes casi idénticas de algo visto en entrenamiento.

En Benchmark cada caso es un archivo independiente, así que se usó una partición estratificada por categoría y etiqueta. En los casos controlados las instancias de una misma plantilla son muy parecidas entre sí, así que la unidad de partición es la plantilla: dentro de cada categoría se reservan plantillas completas para prueba y validación. El resultado es que el conjunto de prueba contiene estructuras de código que el modelo nunca vio, lo que hará la evaluación más exigente pero también más honesta.

Como las plantillas son pocas por categoría, la proporción final se aleja un poco de la recomendada (alrededor de 67/16/17).

In [6]:
pd.crosstab(df.proyecto_origen, df.particion, margins=True, margins_name="total")

particion,test,train,val,total
proyecto_origen,,,,
BenchmarkJava,76,377,82,535
BenchmarkPython,86,375,80,541
casos_controlados,101,279,85,465
total,263,1031,247,1541


In [7]:
grupos_mezclados = df.groupby("grupo")["particion"].nunique().gt(1).sum()
hash_mezclados = df.groupby("sha256")["particion"].nunique().gt(1).sum()
print("Plantillas o casos repartidos en más de una partición:", grupos_mezclados)
print("Códigos idénticos en más de una partición:", hash_mezclados)
(df.particion.value_counts(normalize=True) * 100).round(1)

Plantillas o casos repartidos en más de una partición: 0
Códigos idénticos en más de una partición: 0


particion
train    66.9
test     17.1
val      16.0
Name: proportion, dtype: float64

## Conclusión de la fase

El dataset final tiene 1541 registros, con 772 vulnerables y 769 no vulnerables, en Java, Python y archivos de configuración. Supera los mínimos del enunciado (500 registros y 100 no vulnerables) y cubre las diez categorías de OWASP Top 10:2025 y siete de las diez de OWASP API Security Top 10:2023. Las tres restantes (API6, API9 y API10) no tienen ejemplos porque dependen de la lógica de negocio, del inventario de despliegue o de la confianza en terceros, cosas que no se pueden observar en un archivo aislado; se declaran como no evaluadas.

Las particiones son independientes: ninguna plantilla ni ningún código idéntico aparece en más de una.